In [ ]:
"""******************************************************************************************
* Unitree-G1 RL Sim2Real
*
* Deploy a trained RL locomotion policy (ONNX) on Unitree G1 hardware
*
*     https://github.com/S-CHOI-S/Unitree-G1-Sim2Real.git
*
* Advanced Robot Control Lab. (ARC)
* 	  @ Korea Institute of Science and Technology
*
*	  https://sites.google.com/view/kist-arc
*
******************************************************************************************"""

"* Authors: Sol Choi *"

import os
import time
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from IPython.display import clear_output

today_str = datetime.today().strftime('%Y-%m-%d')

mode = "offline"  # "live" or "offline"

log_path = "../logs/wholebody_rl/log_2025-12-25_16-40-00.csv"
motion_name = "wholebody_rl"

LEFT_LEG = 1
RIGHT_LEG = 2
WAIST = 3
LEFT_ARM = 4
RIGHT_ARM = 5

joint_names = [
  "LeftHipPitch", "LeftHipRoll", "LeftHipYaw", "LeftKnee", "LeftAnklePitch", "LeftAnkleRoll",
  "RightHipPitch", "RightHipRoll", "RightHipYaw", "RightKnee", "RightAnklePitch", "RightAnkleRoll",
  "WaistYaw", "WaistRoll", "WaistPitch",
  "LeftShoulderPitch", "LeftShoulderRoll", "LeftShoulderYaw", "LeftElbow", "LeftWristRoll", "LeftWristPitch", "LeftWristYaw",
  "RightShoulderPitch", "RightShoulderRoll", "RightShoulderYaw", "RightElbow", "RightWristRoll", "RightWristPitch", "RightWristYaw"
]

In [ ]:

def plot_log(df, type, start_time=0, realtime=False):
    if type == LEFT_LEG:
        num_motors = 6
        start_idx = 0
    elif type == RIGHT_LEG:
        num_motors = 6
        start_idx = 6
    elif type == WAIST:
        num_motors = 3
        start_idx = 12
    elif type == LEFT_ARM:
        num_motors = 7
        start_idx = 15
    elif type == RIGHT_ARM:
        num_motors = 7
        start_idx = 22
    
    motor_indices = list(range(num_motors))
    
    fig1, axs1 = plt.subplots(num_motors, 1, figsize=(12, 3 * num_motors), sharex=True)
    fig2, axs2 = plt.subplots(num_motors, 1, figsize=(12, 3 * num_motors), sharex=True)
    t = df["time"]

    for ax in axs1: ax.cla()
    for ax in axs2: ax.cla()

    for i, idx in enumerate(motor_indices):
        # Position: q, qdes, error
        q = df[f"q{idx + start_idx}"]
        qdes = df[f"qdes{idx + start_idx}"]
        error_q = abs(qdes - q)

        axs1[i].plot(t, q, label=f"q{idx + start_idx}", color='blue')
        axs1[i].plot(t, qdes, label=f"qdes{idx + start_idx}", color='red', linestyle='--')
        # axs1[i].plot(t, error_q, label="error", color='green', linestyle=':')
        axs1[i].tick_params(labelbottom=True)
        axs1[i].set_title(f"{joint_names[idx + start_idx]}")
        axs1[i].set_ylabel(f"position")
        axs1[i].xaxis.set_major_locator(MultipleLocator(1.0))
        axs1[i].xaxis.set_minor_locator(MultipleLocator(0.2))
        axs1[i].legend()
        # axs1[i].grid(True)
        axs1[i].grid(True, which='major')
        # axs1[i].grid(True, which='minor', linestyle=':', linewidth=0.5)

        # Torque: tau, taudes, error
        tau = df[f"tau{idx + start_idx}"]
        taudes = df[f"taudes{idx + start_idx}"]
        error_tau = abs(taudes - tau)

        axs2[i].plot(t, tau, label=f"tau{idx + start_idx}", color='purple')
        axs2[i].plot(t, taudes, label=f"taudes{idx + start_idx}", color='orange', linestyle='--')
        # axs2[i].plot(t, error_tau, label="tau error", color='gray', linestyle=':')
        axs2[i].tick_params(labelbottom=True)
        axs2[i].set_title(f"{joint_names[idx + start_idx]}")
        axs2[i].set_ylabel(f"torque")
        axs2[i].xaxis.set_major_locator(MultipleLocator(1.0))
        axs2[i].xaxis.set_minor_locator(MultipleLocator(0.2))
        axs2[i].legend()
        # axs2[i].grid(True)
        axs2[i].grid(True, which='major')
        # axs2[i].grid(True, which='minor', linestyle=':', linewidth=0.5)

        if t.iloc[-1] > 0:
            axs1[i].set_xlim([start_time, t.iloc[-1]])
            axs2[i].set_xlim([start_time, t.iloc[-1]])
        else:
            print(f"Warning: t[-1] = {t.iloc[-1]} is less than {start_time}, skipping xlim.")

    axs1[-1].set_xlabel("Time (s)")
    axs2[-1].set_xlabel("Time (s)")

    # fig1.suptitle("q vs qdes vs error (Joint Tracking)", fontsize=14)
    # fig2.suptitle("tau vs taudes vs error (Torque Tracking)", fontsize=14)

    base_path = f"../logs/{motion_name}/{today_str}"
    os.makedirs(base_path, exist_ok=True)
    
    fig1.tight_layout(rect=[0, 0.03, 1, 0.97])
    # Save fig1 as a separate file
    fig1.savefig(f"{base_path}/onnx_cpp_pos_{motion_name}_{type}.svg", format="svg")
    
    fig2.tight_layout(rect=[0, 0.03, 1, 0.97])
    # Save fig2 as a separate file
    fig2.savefig(f"{base_path}/onnx_cpp_tau_{motion_name}_{type}.svg", format="svg")

    if realtime:
        plt.pause(0.5)
    else:
        plt.show()

if mode == "live":
    plt.ion()
    while True:
        try:
            df = pd.read_csv(log_path)
            clear_output(wait=True)
            plot_log(df, LEFT_LEG, realtime=True)

        except KeyboardInterrupt:
            print("Stopped")
            break
        except Exception as e:
            print("Cannot read log file...", e)
            time.sleep(1)

elif mode == "offline":
    try:
        df = pd.read_csv(log_path)
        start_time = 0
            
        if start_time == 0:
            qdes_ref = df[f"qdes{0}"].to_numpy()
            time_array = df["time"].to_numpy()

            threshold = 0.03

            for j in range(1, len(qdes_ref)):
                if abs(qdes_ref[j] - qdes_ref[j - 1]) > threshold:
                    start_idx_est = max(0, j - 10)
                    start_time = time_array[start_idx_est]
                    print(f"[INFO] Set start_time automatically: {start_time:.3f}s (start at index {j})")
                    break
                
        ## LEG
        plot_log(df, LEFT_LEG, start_time=start_time)
        plot_log(df, RIGHT_LEG, start_time=start_time)
        ## WAIST
        plot_log(df, WAIST, start_time=start_time)
        ## ARM
        plot_log(df, LEFT_ARM, start_time=start_time)
        plot_log(df, RIGHT_ARM, start_time=start_time)
    except Exception as e:
        print("Failed to load log file:", e)